# 06b — A tiny causal transformer — Practice

Read the lesson and run the example. Implement each TODO, then run its checks. A NotImplementedError is expected until you write your answer. Try for 20–30 minutes before opening a hint; open the solution only after recording an attempt. Copy this notebook into work/ to keep your own version.

**Prerequisites:** earlier lessons in the course order. **Environment:** Install requirements/deep.txt. CPU is sufficient.

[Stage guide](../stages/06.md) · [Course map](../PLAN.md) · [Project](../../projects/stage06/README.md)

**Notebook time:** 2–4 hours for an initial pass; project, reading, retrieval practice, and independent transfer use the rest of the stage budget.

Learning objectives:

- Shift language-model targets correctly
- Train a causal attention block
- Measure held-out loss and generate tokens


In [ ]:
from pathlib import Path
import sys

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "datasets" / "financials.csv").exists()), None)
if ROOT is None:
    raise RuntimeError("Open this notebook from inside the learning repository.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

def expect_error(error_type, operation):
    try:
        operation()
    except error_type:
        return
    raise AssertionError(f"Expected {error_type.__name__}")

print("Course root:", ROOT)


## Lesson

Next-token training pairs a sequence x₀…xₜ with targets x₁…xₜ₊₁. If targets are not shifted, a model can learn to copy the current token. Split the text before creating overlapping windows so neighboring chunks cannot leak across evaluation boundaries.

A small causal language model needs token embeddings, positions, masked self-attention, a feed-forward layer, residual paths, normalization, and a vocabulary output projection. Cross entropy compares the logits at each position with the next token. Perplexity is exp(mean cross entropy) under natural logs and a fixed tokenizer; comparing different tokenizers by perplexity can mislead.

Tiny models trained for minutes teach mechanics, not useful general intelligence. Inspect shape invariants, gradient flow, training loss, validation loss, and generation. The project contains a full CPU training script with no downloaded model. Increase scale only after confirming the causal mask and target alignment.


## Worked example

Predict the output before running. Change one input and explain the result.


In [ ]:
import torch
from torch import nn
torch.manual_seed(42)
torch.set_num_threads(1)
sequence = torch.tensor([1, 2, 3, 4])
print(sequence[:-1], sequence[1:])


## Exercise 1: Shifted examples

Return sequence[:-1] and sequence[1:] for a 1D tensor of at least two tokens.

Write down one normal case, one boundary case, and one invalid case before coding.


In [ ]:
def shifted(sequence):
    raise NotImplementedError


In [ ]:
x, y = shifted(torch.tensor([3, 1, 4]))
assert x.tolist() == [3, 1] and y.tolist() == [1, 4]
print("Exercise 1: checks passed")


<details><summary>Hint — open after trying</summary>

Every target is one position ahead.

</details>


## Exercise 2: Future mask

Return an n×n boolean mask where True blocks future keys; diagonal stays False.

Write down one normal case, one boundary case, and one invalid case before coding.


In [ ]:
def causal_mask(n):
    raise NotImplementedError


In [ ]:
mask = causal_mask(3)
assert mask.tolist() == [[False, True, True], [False, False, True], [False, False, False]]
print("Exercise 2: checks passed")


<details><summary>Hint — open after trying</summary>

Boolean mask semantics depend on the API; this is MultiheadAttention’s convention.

</details>


## Exercise 3: Transformer block

Implement TinyBlock with batch_first MultiheadAttention(d,2), LayerNorm, and residual connection. Forward accepts (N,T,d), uses causal_mask, returns same shape.

Write down one normal case, one boundary case, and one invalid case before coding.


In [ ]:
class TinyBlock(nn.Module):
    def __init__(self, d=8):
        super().__init__()
        raise NotImplementedError
    def forward(self, x):
        raise NotImplementedError


In [ ]:
block = TinyBlock().eval()
x = torch.randn(1, 4, 8)
changed = x.clone(); changed[:, 3] += 10
assert block(x).shape == x.shape
assert torch.allclose(block(x)[:, :3], block(changed)[:, :3], atol=1e-6)
print("Exercise 3: checks passed")


<details><summary>Hint — open after trying</summary>

Add the original input before normalization.

</details>


## Independent transfer

Train projects/stage06/tiny_lm.py, add a second block, compare held-out loss and generation, then implement cached inference as an extension. Explain why train loss alone is insufficient.

Record your implementation, errors, measurements, and explanation in work/. See the project’s ANSWERS.md after attempting the brief.


## Explain without code

1. Why split text before windowing?

2. Why a residual path?


## Reading and review

- [Primary reference 1](https://docs.pytorch.org/docs/stable/generated/torch.nn.MultiheadAttention.html)

After 2, 7, and 30 days: explain the concept from memory, solve a changed-input version, and log any gaps in progress.md.
